In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='01f12e1a-e4f8-4dd8-9d49-7678ee6f834f'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='0a2aa0fb-fc5a-4908-8f0c-27f7ac753dd3'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='f2ee0f38-c247-430d-9ca0-e45435eaf97c', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='717162b6-f547-472b-a98f-15dd66eb0995'), AIMessage(content='好的，我来帮你查一下北京今天的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 316, 'total_tokens': 370, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 316}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_pro

In [3]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

好的，我来帮你查一下北京今天的天气情况。
Tool Calls:
  get_weather (call_00_lLhJighYQG0fX7qVorIl0129)
 Call ID: call_00_lLhJighYQG0fX7qVorIl0129
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

北京今天天气是**晴天（sunny）**☀️，是个阳光明媚的好天气，适合外出活动或者晾晒衣物哦！

还有什么需要帮忙的吗，虎哥？😊


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [4]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.7-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [5]:
# 创建Agent
agent = create_agent(model=model)

In [6]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [7]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张图片描绘了一个温馨、宁静的海滩场景，主要展示了人与宠物之间的亲密互动。以下是详细的画面内容描述：

**1. 主要人物与动物：**
*   **年轻女子**：画面右侧坐着一位长发女子。她身穿蓝白格子的长袖衬衫和深色裤子，赤脚坐在沙滩上。她面带灿烂的微笑，神情专注且快乐地看着对面的狗。
*   **狗**：画面左侧坐着一只浅黄色的大型犬（看起来像拉布拉多犬）。它身上佩戴着带有彩色图案的胸背带，姿态端正。

**2. 动作与互动：**
*   **握手/击掌**：这是画面的核心动作。女子伸出左手手掌，狗狗顺从地抬起右前爪搭在她的手上，两人正在完成一个“握手”的动作。
*   **奖励**：女子的右手似乎捏着一个小东西（可能是零食），正准备奖励狗狗，这解释了狗狗为何如此配合。
*   **牵引绳**：在狗狗的身后，沙滩上随意放置着一根红色的牵引绳，表明它们是来散步的。

**3. 背景与环境：**
*   **海滩与大海**：背景是广阔的海面，可以看到白色的海浪正在涌向岸边。沙滩看起来比较宽阔，沙质细腻，表面有一些自然的纹理和脚印。
*   **光线与氛围**：图片右侧有强烈的阳光照射进来（看起来像是日落或日出时分），形成了温暖的逆光效果。阳光给女子的头发和轮廓镀上了一层金色的光晕，整个画面充满了温暖、和谐与治愈的氛围。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [8]:
!uv add ipywidgets

Resolved 197 packages in 5ms
Checked 191 packages in 6ms


然后，我们创建一个上传组件，用于模拟图片上传。


In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [10]:
print(uploader.value)

()


In [11]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

IndexError: tuple index out of range

In [ ]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中内容"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)